# v0.37.0 — Hybrid search

## 1. Intro

Since v0.37.0 a queryset ranks its records by two searches at once, with the full SurrealDB-ORM's `hybrid_search()`:

- the records nearest to a query vector, as `similar_to()` finds them (v0.36.0), and the best matches of a text, as
  `search()` ranks them by BM25 score (v0.35.0), are fused into one ranking by Reciprocal Rank Fusion: each record
  gets 1 / (`rrf_k` + rank) from each list it is in, its rank counted from 1 (`rrf_k` is 60 by default);
- the arguments are the full ORM's, keyword-only — `vector_field`, `vector`, `vector_limit=20`, `text_field`,
  `text_query`, `text_limit=20`, `rrf_k=60` —, with `ef=` and `dist=` as in `similar_to()`;
- each record comes back as a dict, as in the full ORM: its fields, `_rrf_score`, `_knn_distance` and
  `_search_score`;
- the queryset applies: its conditions select the records of both lists, and `select()`, `fetch()`,
  `limit()`/`offset()`, `cache()`, transactions and named connections work as for `exec()`.

SurrealDB 3.x fuses with `search::rrf()`; on 2.x and the embedded engine the ORM fuses with the same formula — the
same two lists give the same results. The notebook runs top to bottom on both lines. The outputs below come from a
SurrealDB 3.x server: a 2.x server prints the same records, distances and scores, and in §9 the sentence of its line.

## 2. Connect

`SURREALDB_HOST` and `SURREALDB_PORT` choose the server (default `localhost:8000`), over WebSocket.

In [1]:
import contextlib
import os

from surreal_orm_lite import SurrealDBConnectionManager

HOST = os.environ.get("SURREALDB_HOST", "localhost")
PORT = os.environ.get("SURREALDB_PORT", "8000")
# WebSocket (.../rpc), as the other notebooks.
SurrealDBConnectionManager.set_connection(
    url=f"ws://{HOST}:{PORT}/rpc",
    user="root",
    password="root",
    namespace="examples",
    database="examples",
)
print("Connection configured:", SurrealDBConnectionManager.is_connection_set())

Connection configured: True


## 3. Eight passages

`Passage` is the model of the README's section 38. Its `embedding` is a `VectorField[3]` holding a hand-made embedding
of three numbers — one per topic: pets, code, home — so the notebook downloads no model; an embedding model gives
hundreds of numbers, used the same way. `hybrid_search()` needs both of the model's indexes: `passage_embedding`, an
HNSW vector index with the `COSINE` distance, for the vector search, and `passage_title`, a full-text index on
`title` with the analyzer `words`, for the full-text search.

The cell first removes what an earlier run left — the table, with its indexes, then the analyzer — so the notebook can
be re-executed in place. `define_table()` then defines the table, and `define_indexes()` the analyzer, then the
indexes, before the eight passages are saved.

In [2]:
from surreal_orm_lite import Analyzer, BaseSurrealModel, Index, SurrealConfigDict, VectorField

WORDS = Analyzer("words", ["blank"], ["lowercase"])


class Passage(BaseSurrealModel):
    model_config = SurrealConfigDict(
        table_name="passage",
        indexes=[
            Index("passage_embedding", "embedding", hnsw=True, dimension=3, dist="cosine"),
            Index("passage_title", "title", search_analyzer=WORDS),
        ],
    )

    id: str | None = None
    title: str
    topic: str = ""
    embedding: VectorField[3]


client = await SurrealDBConnectionManager.get_client()
for statement in ("REMOVE TABLE IF EXISTS passage;", "REMOVE ANALYZER IF EXISTS words;"):
    with contextlib.suppress(Exception):
        await client.query(statement, {})

await Passage.define_table()
await Passage.define_indexes()
for key, title, topic, embedding in [
    ("p1", "Cats and dogs", "pets", [1.0, 0.0, 0.0]),
    ("p2", "Cats at home", "pets", [0.9, 0.1, 0.0]),
    ("p3", "Rust programming", "tech", [0.0, 1.0, 0.0]),
    ("p4", "Python programming", "tech", [0.1, 0.9, 0.0]),
    ("p5", "Gardening", "home", [0.0, 0.0, 1.0]),
    ("p6", "Cats and code", "news", [0.5, 0.5, 0.0]),
    ("p7", "Rust and Python programming", "tech", [0.2, 0.8, 0.0]),
    ("p8", "Cooking", "home", [0.0, 0.2, 0.8]),
]:
    await Passage(id=key, title=title, topic=topic, embedding=embedding).save()
print("passages:", await Passage.objects().count())

passages: 8


## 4. The two rankings

Each search alone first. `search(title="programming")` keeps the three passages whose title holds "programming", and
`SearchScore(0)` gives each its BM25 score: the two two-word titles score alike, above the four-word one, so they are
sorted by score, then id. `similar_to("embedding", pets, 3)` keeps the three passages nearest to `pets`, nearest
first, each with its distance as `_knn_distance`. `hybrid_search()` runs both searches and fuses their lists.

In [3]:
from surreal_orm_lite import SearchScore

pets = [1.0, 0.0, 0.0]
code = [0.0, 1.0, 0.0]

print("The best matches of 'programming':")
matches = await Passage.objects().search(title="programming").annotate(score=SearchScore(0)).order_by("-score", "id").exec()
for passage in matches:
    print(" ", passage.id, passage.title, round(passage.score, 4))
print("The nearest to pets:")
for passage in await Passage.objects().similar_to("embedding", pets, 3).exec():
    print(" ", passage.id, passage.title, round(passage._knn_distance, 4))

The best matches of 'programming':
  p3 Rust programming 0.4832
  p4 Python programming 0.4832
  p7 Rust and Python programming 0.3531
The nearest to pets:
  p1 Cats and dogs 0.0
  p2 Cats at home 0.0061
  p6 Cats and code 0.2929


## 5. Fuse them

`hybrid_search()` takes the `vector_limit` passages nearest to `pets` and the `text_limit` best matches of
"programming" — three each here — and fuses them: `p1`, first of the vector list, and `p3`, first of the full-text
list, both score 1/61; `p2` and `p4`, second, 1/62; `p6` and `p7`, third, 1/63. A record in both lists adds its two
terms (§7).

The result is a list of dicts — the full ORM's return type —, never model instances: each dict holds the record's
fields, keyed by their Python names as `exec()`'s dict rows are, and `_rrf_score`, `_knn_distance` (`None` for a record
only the full-text search found) and `_search_score` (`None` for one only the vector search found).

Equal scores have a fixed order: the vector list's records first, in its order, then the records the full-text list
alone found, in its order. SurrealDB's `search::rrf()` alone returns equal scores in no fixed order; `hybrid_search()`
gives the same two lists the same order on every line and every call, so a page of them never changes (§6). Each list
is SurrealDB's own, though: BM25 scores depend on the line for a term found in more than half the documents, and an
exact scan on 2.x can pick a different vector list from one call to the next (§8).

In [4]:
def four_decimals(value: float | None) -> str:
    return "None" if value is None else f"{value:.4f}"


results = await Passage.objects().hybrid_search(
    vector_field="embedding", vector=pets, vector_limit=3,
    text_field="title", text_query="programming", text_limit=3,
)
print(type(results).__name__, "of", type(results[0]).__name__)
for row in results:
    print(row["id"], row["title"], f"{row['_rrf_score']:.4f}", four_decimals(row["_knn_distance"]), four_decimals(row["_search_score"]))
print("p1 scores 1/61:", results[0]["_rrf_score"] == 1 / 61)

list of dict
passage:p1 Cats and dogs 0.0164 0.0000 None
passage:p3 Rust programming 0.0164 None 0.4832
passage:p2 Cats at home 0.0161 0.0061 None
passage:p4 Python programming 0.0161 None 0.4832
passage:p6 Cats and code 0.0159 0.2929 None
passage:p7 Rust and Python programming 0.0159 None 0.3531
p1 scores 1/61: True


## 6. Paging

`limit()` and `offset()` page through the fused records — at most `vector_limit + text_limit` of them —, best
first.

In [5]:
arguments = {
    "vector_field": "embedding",
    "vector": pets,
    "vector_limit": 3,
    "text_field": "title",
    "text_query": "programming",
    "text_limit": 3,
}
print("limit(3):", [str(row["id"]) for row in await Passage.objects().limit(3).hybrid_search(**arguments)])
print("offset(1).limit(2):", [str(row["id"]) for row in await Passage.objects().offset(1).limit(2).hybrid_search(**arguments)])

limit(3): ['passage:p1', 'passage:p3', 'passage:p2']
offset(1).limit(2): ['passage:p3', 'passage:p2']


## 7. With conditions

The queryset's conditions select the records of both lists: with `filter(topic="tech")`, the vector search keeps the
tech passages nearest to `code`, and the full-text search the tech passages matching "programming". The three tech
passages are in both lists, at the same rank, so each adds its two terms: 2/61, 2/62 and 2/63.

In [6]:
tech = await Passage.objects().filter(topic="tech").hybrid_search(
    vector_field="embedding", vector=code, vector_limit=3,
    text_field="title", text_query="programming", text_limit=3,
)
for row in tech:
    print(row["id"], row["title"], f"{row['_rrf_score']:.4f}")

passage:p3 Rust programming 0.0328
passage:p4 Python programming 0.0323
passage:p7 Rust and Python programming 0.0317


## 8. An exact scan and `select()`

`dist="cosine"` scans every record exactly instead of searching the HNSW index, as in `similar_to()`: here the same
three nearest passages, so the same fused ranking. With records at an equal distance at the `vector_limit`-th place
— none here —, identical calls still get the same vector list on every line: SurrealDB 2.x's and the embedded
engine's own scan picks among them in no fixed order, so there the ORM ranks it by distance, then id, and keeps the
tied records by id. `select("title")` reads the title only: each dict holds `id`, always read, `title` and the three
values of `hybrid_search()`.

In [7]:
exact = await Passage.objects().select("title").hybrid_search(**arguments, dist="cosine")
print(sorted(exact[0]))
print([str(row["id"]) for row in exact])

['_knn_distance', '_rrf_score', '_search_score', 'id', 'title']
['passage:p1', 'passage:p3', 'passage:p2', 'passage:p4', 'passage:p6', 'passage:p7']


## 9. The line

SurrealDB 3.x has `search::rrf()`: the ORM sends both lists in one statement, and SurrealDB fuses them. SurrealDB 2.x
and the embedded engine have no such function: the ORM reads both lists in one statement and fuses them itself, with
the same formula, summed in the same order. The ORM reads the server's version once per connection.

In [8]:
version = await (await SurrealDBConnectionManager.get_client()).version()
major = int(version.rsplit("-", 1)[-1].split(".")[0])  # "surrealdb-3.3.4" -> 3
print(f"Server: SurrealDB {major}.x")
if major >= 3:
    print("SurrealDB's search::rrf() fused the lists")
else:
    print("the ORM fused the lists (SurrealDB 2.x has no search::rrf())")
print("The results above are identical either way.")

Server: SurrealDB 3.x
SurrealDB's search::rrf() fused the lists
The results above are identical either way.


## 10. What is refused

Each before any request. `hybrid_search()` runs its own searches and orders the records by their fused score, so a
queryset holding `order_by()`, `annotate()`, `values()`, `search()` or `similar_to()` raises `ValueError`. `rrf_k`
must be from 0 to 10 000, and `vector_limit` and `text_limit` from 1 to 10 000: another value raises `ValueError`
too.

In [9]:
try:
    await Passage.objects().order_by("title").hybrid_search(
        vector_field="embedding", vector=pets, text_field="title", text_query="cats",
    )
except ValueError as error:
    print(error)
print()
try:
    await Passage.objects().hybrid_search(**arguments, rrf_k=-1)
except ValueError as error:
    print(error)

hybrid_search() cannot be combined with order_by(): it runs its own vector and full-text searches and orders the records by their fused score. Remove them from the queryset.

hybrid_search() rrf_k must be between 0 and 10000, got -1.


## 11. Cleanup

The table goes with its indexes, then the analyzer, and the connection is closed.

In [10]:
await client.query("REMOVE TABLE IF EXISTS passage;", {})
print(await SurrealDBConnectionManager.remove_analyzer("words"))
await SurrealDBConnectionManager.close_connection()
print("Done")

REMOVE ANALYZER IF EXISTS `words`;
Done
